# Scrambling example

In this example we demonstrate how to use the data scrambling mechanism of SkyLLH.

In [1]:
# Imports and helper functions for examples.

import numpy as np

from skyllh.core.config import Config
from skyllh.core.dataset import Dataset
from skyllh.core.livetime import Livetime
from skyllh.core.random import RandomStateService
from skyllh.core.scrambling import DataScrambler, UniformRAScramblingMethod
from skyllh.core.times import LivetimeTimeGenerationMethod, TimeGenerator
from skyllh.i3.scrambling import I3TimeScramblingMethod

# Create default configuration.
cfg = Config()
dataset = Dataset(
    cfg=cfg,
    name='example',
    exp_pathfilenames=None,
    mc_pathfilenames=None,
    livetime=None,
    default_sub_path_fmt='',
    version=1,
)


def gen_data(rss, N):
    """Create uniformly distributed data on sphere."""
    arr = np.empty(
        (N,),
        dtype=[
            ('azi', np.float64),
            ('zen', np.float64),
            ('ra', np.float64),
            ('dec', np.float64),
            ('time', np.float64),
        ],
    )

    arr['ra'] = rss.random.uniform(0.0, 2.0 * np.pi, N)
    arr['dec'] = rss.random.uniform(-np.pi, np.pi, N)

    return arr

## Example 1

In [2]:
def ex1():
    """Data scrambling via right-ascention scrambling."""
    print('Example 1')
    print('=========')

    rss = RandomStateService(seed=1)

    # Generate some pseudo data.
    data = gen_data(rss=rss, N=10)
    print(f'before scrambling: data["ra"]={data["ra"]}')

    # Create DataScrambler instance with uniform RA scrambling.
    scrambler = DataScrambler(method=UniformRAScramblingMethod())

    # Scramble the data.
    scrambler.scramble_data(rss=rss, dataset=dataset, data=data)

    print(f'after scrambling: data["ra"]={data["ra"]}')


ex1()

Example 1
before scrambling: data["ra"]=[2.62022653e+00 4.52593227e+00 7.18638172e-04 1.89961158e+00
 9.22094457e-01 5.80180502e-01 1.17030742e+00 2.17122208e+00
 2.49296356e+00 3.38548539e+00]
after scrambling: data["ra"]=[5.03122651 6.08376691 1.96930219 4.34999129 5.50651545 5.62097944
 0.53434854 0.24538844 1.067076   5.51753208]


## Example 2

In [3]:
def ex2():
    """Data scrambling via detector on-time scrambling."""
    print('Example 2')
    print('=========')

    rss = RandomStateService(seed=1)

    # Generate some pseudo data.
    data = gen_data(rss=rss, N=10)
    print(f'before scrambling: data["ra"]={data["ra"]}')

    # Create a Livetime object, which defines the detector live-time.
    lt = Livetime(uptime_mjd_intervals_arr=np.array([[55000, 56000], [60000, 69000]], dtype=np.float64))

    # Create a TimeGenerator with an on-time time generation method.
    timegen = TimeGenerator(method=LivetimeTimeGenerationMethod(livetime=lt))

    # Create DataScrambler with IceCube time scrambing method.
    scrambler = DataScrambler(method=I3TimeScramblingMethod(timegen))

    # Scramble the data.
    scrambler.scramble_data(rss=rss, dataset=dataset, data=data)

    print(f'after scrambling: data["ra"]={data["ra"]}')


ex2()

Example 2
before scrambling: data["ra"]=[2.62022653e+00 4.52593227e+00 7.18638172e-04 1.89961158e+00
 9.22094457e-01 5.80180502e-01 1.17030742e+00 2.17122208e+00
 2.49296356e+00 3.38548539e+00]
after scrambling: data["ra"]=[1.95150284 0.42176583 4.80220088 0.7701056  5.19938553 3.15038276
 4.77616435 3.81213287 5.62496285 2.56997617]
